# 자녀 가치관 기반 유저 매칭 추천 시스템

이 노트북은 설문조사 데이터를 기반으로 선호동물상 외관과 자녀 가치관을 분석하여  
**유사도 기반 추천**과 **보완도 기반 추천**을 제공하는 매칭 시스템을 구현합니다.

---
## 목차
1. 데이터 불러오기
2. 데이터 전처리
3. 피처 엔지니어링 (One-hot, 중요도 가중치, MBTI형 성향 점수)
4. 유사도 기반 매칭 추천
5. 보완도 기반 매칭 추천
6. 통합 추천 시스템
---

In [4]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기

In [5]:
# CSV 파일 경로 설정 (필요시 수정)
FILE_PATH = "./data/저출산 소개팅_설문조사_시나리오추가.csv"

df = pd.read_csv(FILE_PATH)
print(f"데이터 shape: {df.shape}")
df.head()

데이터 shape: (34, 24)


,타임스탬프,0. 당신의 성함,0. 당신의 이상형,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.,...,맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도",Unnamed: 23
0,2026/02/27 3:53:18 PM GMT+9,강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,...,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3,https://drive.google.com/u/0/open?usp=forms_we...
1,2026/02/27 4:15:15 PM GMT+9,임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,...,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3,NaN
2,2026/02/27 4:19:32 PM GMT+9,이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,...,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4,NaN
3,2026/02/27 4:29:59 PM GMT+9,쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,...,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2,NaN
4,2026/02/27 4:30:18 PM GMT+9,김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,...,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 24 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   타임스탬프                                                                                  34 non-null     str  
 1   0. 당신의 성함                                                                              34 non-null     str  
 2   0. 당신의 이상형                                                                             34 non-null     str  
 3   1-1. 희망하는 자녀 수                                                                         34 non-null     str  
 4   1-2. 희망하는 자녀 구성                                                                        34 non-null     str  
 5   1-3. 자녀 갖고 싶은 시기                                                                       34 non-null     st

In [7]:
# 원본 데이터 백업
df_raw = df.copy()

# 2. 데이터 전처리

## (1) 컬럼명 정리

In [8]:
def clean_colname(col: str) -> str:
    """컬럼명 정규화: 공백/따옴표/줄바꿈 정리"""
    col = str(col).strip()
    col = col.replace("\n", " ").replace("\t", " ")
    col = re.sub(r"\s+", " ", col)
    col = col.replace('"', "")
    return col

# 정규화 적용
cleaned_cols = [clean_colname(c) for c in df.columns]

# 중복 컬럼명 처리
seen = {}
unique_cols = []
for c in cleaned_cols:
    if c in seen:
        seen[c] += 1
        unique_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        unique_cols.append(c)

df.columns = unique_cols
print("정리된 컬럼 목록:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

정리된 컬럼 목록:
  0: 타임스탬프
  1: 0. 당신의 성함
  2: 0. 당신의 이상형
  3: 1-1. 희망하는 자녀 수
  4: 1-2. 희망하는 자녀 구성
  5: 1-3. 자녀 갖고 싶은 시기
  6: 1-4. 생물학적 출산이 어려움 발생 시 대안
  7: 1. 자녀 계획 및 가족 구성 항목에 대해 중요도
  8: 아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?
  9: 평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.
  10: 경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?
  11: 재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?
  12: 두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?
  13: 한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.
  14: 맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?
  15: 양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?
  16: 주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?
  17: 아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?
  18: 4-1. 자녀 교육비/양육비 지출 비중
  19: 4-2. 육아 휴직, 양육 부담
  20: 4. 경제적 지원 및 가사 분담에 대해 중요도
  21: 5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?
  22: 5. 자녀 가치관에 대해 중요도
  23: Unnamed: 23


## (2) 컬럼 리네임

In [9]:
# 컬럼 리네임 매핑
RENAME_MAP = {
    '0. 당신의 성함': 'user_name',
    '0. 당신의 이상형': 'ideal_type',
    '1-1. 희망하는 자녀 수': 'p_children_count',
    '1-2. 희망하는 자녀 구성': 'p_children_composition',
    '1-3. 자녀 갖고 싶은 시기': 'p_children_timing',
    '1-4. 생물학적 출산이 어려움 발생 시 대안': 'p_infertility_alternative',
    '1. 자녀 계획 및 가족 구성 항목에 대해 중요도': 'imp_family_plan',
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?': 'sc_toothbrushing',
    '평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.': 'sc_bedtime_story',
    '경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?': 'sc_competition_2nd',
    '재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?': 'sc_talent_education',
    '두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?': 'sc_discipline_conflict',
    '한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.': 'sc_play_vs_chores',
    '맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?': 'sc_grandparents_help',
    '양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?': 'sc_inlaws_advice',
    '주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?': 'sc_rainy_zoo',
    '아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?': 'sc_education_fund_risk',
    '4-1. 자녀 교육비/양육비 지출 비중': 'e_childcare_cost_share',
    '4-2. 육아 휴직, 양육 부담': 'e_parental_leave_burden',
    '4. 경제적 지원 및 가사 분담에 대해 중요도': 'imp_econ_housework',
    '5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?': 'child_values_open',
    '5. 자녀 가치관에 대해 중요도': 'imp_child_values'
}

# 정확한 매칭이 안 되는 경우를 위한 부분 매칭
def find_and_rename(df, rename_map):
    new_rename = {}
    for old_name, new_name in rename_map.items():
        # 정확한 매칭 시도
        if old_name in df.columns:
            new_rename[old_name] = new_name
        else:
            # 부분 매칭 시도 (핵심 키워드로)
            for col in df.columns:
                # 핵심 키워드 추출하여 매칭
                key_parts = old_name.split()[:3]  # 앞 3단어
                if all(part in col for part in key_parts[:2]):
                    new_rename[col] = new_name
                    break
    return new_rename

actual_rename = find_and_rename(df, RENAME_MAP)
df = df.rename(columns=actual_rename)

print(f"리네임된 컬럼 수: {len(actual_rename)}")
print("\n현재 컬럼 목록:")
print(list(df.columns))

리네임된 컬럼 수: 22

현재 컬럼 목록:
['타임스탬프', 'user_name', 'ideal_type', 'p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative', 'imp_family_plan', 'sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk', 'e_childcare_cost_share', 'e_parental_leave_burden', 'imp_econ_housework', 'child_values_open', 'imp_child_values', 'Unnamed: 23']


## (3) 불필요한 컬럼 제거 및 인덱스 설정

In [10]:
# 타임스탬프 및 불필요한 컬럼 제거
drop_cols = ['타임스탬프']
# Unnamed 컬럼도 제거
drop_cols += [c for c in df.columns if 'Unnamed' in c]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# user_name을 인덱스로 설정
if 'user_name' in df.columns:
    df = df.set_index('user_name')

print(f"최종 데이터 shape: {df.shape}")
df.head()

최종 데이터 shape: (34, 21)


,ideal_type,p_children_count,p_children_composition,p_children_timing,p_infertility_alternative,imp_family_plan,sc_toothbrushing,sc_bedtime_story,sc_competition_2nd,sc_talent_education,...,sc_play_vs_chores,sc_grandparents_help,sc_inlaws_advice,sc_rainy_zoo,sc_education_fund_risk,e_childcare_cost_share,e_parental_leave_burden,imp_econ_housework,child_values_open,imp_child_values
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,5,4,...,4,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3
임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,4,2,...,4,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3
이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,5,4,...,5,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4
쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,5,4,...,5,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2
김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,4,2,...,1,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3


## (4) 컬럼 그룹 정의

In [11]:
# 컬럼 그룹 정의
META_COLS = ['ideal_type']

PLAN_COLS = ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']

SCENARIO_COLS = ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education',
                 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 
                 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']

ECON_COLS = ['e_childcare_cost_share', 'e_parental_leave_burden']

TEXT_COLS = ['child_values_open']

IMPORTANCE_COLS = ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']

# 존재하는 컬럼만 필터링
all_groups = {
    'META_COLS': [c for c in META_COLS if c in df.columns],
    'PLAN_COLS': [c for c in PLAN_COLS if c in df.columns],
    'SCENARIO_COLS': [c for c in SCENARIO_COLS if c in df.columns],
    'ECON_COLS': [c for c in ECON_COLS if c in df.columns],
    'TEXT_COLS': [c for c in TEXT_COLS if c in df.columns],
    'IMPORTANCE_COLS': [c for c in IMPORTANCE_COLS if c in df.columns]
}

for gname, cols in all_groups.items():
    print(f"{gname}: {len(cols)}개 - {cols}")

META_COLS: 1개 - ['ideal_type']
PLAN_COLS: 4개 - ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']
SCENARIO_COLS: 10개 - ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']
ECON_COLS: 2개 - ['e_childcare_cost_share', 'e_parental_leave_burden']
TEXT_COLS: 1개 - ['child_values_open']
IMPORTANCE_COLS: 3개 - ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']


# 3. 피처 엔지니어링

## (1) 범주형 변수 One-Hot Encoding (중요도 가중치 적용)

In [12]:
def one_hot_with_importance(df, col, importance_col=None, known_categories=None):
    """
    범주형 컬럼을 One-Hot 인코딩하고, 중요도 가중치를 적용합니다.
    복수 응답(세미콜론 구분)도 처리합니다.
    
    Parameters:
    -----------
    df : DataFrame
    col : str - 인코딩할 컬럼명
    importance_col : str - 중요도 컬럼명 (None이면 가중치 1)
    known_categories : list - 알려진 카테고리 목록 (그 외는 '기타'로 처리)
    
    Returns:
    --------
    DataFrame with one-hot encoded columns
    """
    if col not in df.columns:
        return pd.DataFrame(index=df.index)
    
    # 복수 응답 분리 및 카테고리 수집
    all_categories = set()
    for val in df[col].dropna():
        for item in str(val).split(';'):
            item = item.strip()
            if item:
                all_categories.add(item)
    
    # 알려진 카테고리가 있으면 그 외는 '기타'로
    if known_categories:
        final_categories = list(known_categories) + ['기타']
    else:
        final_categories = list(all_categories)
    
    # One-Hot 인코딩 결과 초기화
    result = pd.DataFrame(0.0, index=df.index, columns=[f"{col}_{cat}" for cat in final_categories])
    
    # 중요도 가중치 (1~5점을 0.2~1.0으로 변환)
    if importance_col and importance_col in df.columns:
        weights = df[importance_col].fillna(3) / 5.0
    else:
        weights = pd.Series(1.0, index=df.index)
    
    # 각 행에 대해 One-Hot 인코딩
    for idx in df.index:
        val = df.loc[idx, col]
        if pd.isna(val):
            continue
        
        items = [item.strip() for item in str(val).split(';') if item.strip()]
        n_items = len(items)
        
        for item in items:
            if known_categories and item not in known_categories:
                col_name = f"{col}_기타"
            else:
                col_name = f"{col}_{item}"
            
            if col_name in result.columns:
                # 복수 응답인 경우 가중치를 분배
                result.loc[idx, col_name] = weights[idx] / n_items
    
    return result

In [13]:
# 알려진 카테고리 정의
KNOWN_CATEGORIES = {
    'p_children_count': ['1명', '2명', '3명', '그 이상'],
    'p_children_composition': ['오직 딸', '오직 아들', '딸 1명, 아들 1명'],
    'p_children_timing': ['결혼 즉시', '결혼 후 1~2년 이내', '결혼 후 3~5년 이내', '경제적 안정 후'],
    'p_infertility_alternative': ['의학적 도움 적극 시도', '입양 고려', '무자녀'],
    'e_childcare_cost_share': ['노후보단 자녀 교육', '노후 먼저, 남는 예산으로 지원'],
    'e_parental_leave_burden': ['경제력 높은 사람 일하고, 한명은 전담 육아', '맞벌이하면서 외부 도움(조부모, 시터)'],
    'child_values_open': ['경제적 성공, 사회적 지위', '도덕적, 타인 배려', '자신이 좋아하는 일, 행복', '회복탄력성, 생활력 강한 사람']
}

# 중요도 매핑
IMPORTANCE_MAPPING = {
    'p_children_count': 'imp_family_plan',
    'p_children_composition': 'imp_family_plan',
    'p_children_timing': 'imp_family_plan',
    'p_infertility_alternative': 'imp_family_plan',
    'e_childcare_cost_share': 'imp_econ_housework',
    'e_parental_leave_burden': 'imp_econ_housework',
    'child_values_open': 'imp_child_values'
}

# One-Hot Encoding 수행
onehot_dfs = []
for col, categories in KNOWN_CATEGORIES.items():
    if col in df.columns:
        imp_col = IMPORTANCE_MAPPING.get(col)
        oh_df = one_hot_with_importance(df, col, imp_col, categories)
        onehot_dfs.append(oh_df)
        print(f"{col}: {oh_df.shape[1]}개 피처 생성")

# 모든 One-Hot 컬럼 합치기
df_onehot = pd.concat(onehot_dfs, axis=1)
print(f"\n총 One-Hot 피처 수: {df_onehot.shape[1]}")
df_onehot.head()

p_children_count: 5개 피처 생성
p_children_composition: 4개 피처 생성
p_children_timing: 5개 피처 생성
p_infertility_alternative: 4개 피처 생성
e_childcare_cost_share: 3개 피처 생성
e_parental_leave_burden: 3개 피처 생성
child_values_open: 5개 피처 생성

총 One-Hot 피처 수: 29


,p_children_count_1명,p_children_count_2명,p_children_count_3명,p_children_count_그 이상,p_children_count_기타,p_children_composition_오직 딸,p_children_composition_오직 아들,"p_children_composition_딸 1명, 아들 1명",p_children_composition_기타,p_children_timing_결혼 즉시,...,"e_childcare_cost_share_노후 먼저, 남는 예산으로 지원",e_childcare_cost_share_기타,"e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아","e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)",e_parental_leave_burden_기타,"child_values_open_경제적 성공, 사회적 지위","child_values_open_도덕적, 타인 배려","child_values_open_자신이 좋아하는 일, 행복","child_values_open_회복탄력성, 생활력 강한 사람",child_values_open_기타
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.6,0.0,0.0
임경빈,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,...,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.6,0.0
이정미,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,...,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.8,0.0
쳌,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,...,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.4,0.0,0.0
김용희,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,...,0.0,0.0,0.0,0.4,0.0,0.0,0.0,0.6,0.0,0.0


## (2) 이상형 동물상 One-Hot Encoding

In [14]:
# 이상형 동물상 One-Hot (가중치 없음, 단순 매칭용)
KNOWN_IDEAL_TYPES = ['강아지상', '고양이상', '사슴상', '토끼상', '꼬부기상', '곰상', '쿼카상', 
                      '쥐상', '도룡뇽상', '여우상', '너구리상', '오리상', '꽃돼지상']

if 'ideal_type' in df.columns:
    df_ideal = pd.get_dummies(df['ideal_type'], prefix='ideal_type')
    # 알려지지 않은 동물상 처리
    for animal in KNOWN_IDEAL_TYPES:
        col_name = f'ideal_type_{animal}'
        if col_name not in df_ideal.columns:
            df_ideal[col_name] = 0
    print(f"이상형 동물상 피처 수: {df_ideal.shape[1]}")
else:
    df_ideal = pd.DataFrame(index=df.index)

df_ideal.head()

이상형 동물상 피처 수: 13


,ideal_type_강아지상,ideal_type_고양이상,ideal_type_곰상,ideal_type_꼬부기상,ideal_type_꽃돼지상,ideal_type_너구리상,ideal_type_여우상,ideal_type_오리상,ideal_type_쿼카상,ideal_type_토끼상,ideal_type_사슴상,ideal_type_쥐상,ideal_type_도룡뇽상
user_name,,,,,,,,,,,,,
강현준,False,False,False,False,True,False,False,False,False,False,0,0,0
임경빈,False,False,False,True,False,False,False,False,False,False,0,0,0
이정미,False,False,True,False,False,False,False,False,False,False,0,0,0
쳌,False,True,False,False,False,False,False,False,False,False,0,0,0
김용희,False,False,False,False,False,False,True,False,False,False,0,0,0


## (3) 시나리오 응답 기반 MBTI형 성향 점수 계산

5개 축:
- 양육 실행 스타일 :    S (Structure: 규칙·일관성)	F (Freedom: 자율·유연성)
- 교육/성장 우선순위 :  A (Achievement: 성취·경쟁)	H (Holistic: 정서·균형)
- 공동양육 운영 방식 :  E (Equal: 공동분담)	L (Lead: 한명 주도)
- 확장가족/경계 :       B (Boundary: 경계·부부결정)	T (Tribe: 가족개입/공동체)
- 계획/리스크 대응 :    P (Planned: 계획·안정)	G (Go-with-flow: 적응·즉흥)

In [15]:
# 시나리오 컬럼이 존재하는지 확인
scenario_cols = [c for c in SCENARIO_COLS if c in df.columns]
print(f"찾은 시나리오 컬럼 수: {len(scenario_cols)}")
print(scenario_cols)

찾은 시나리오 컬럼 수: 10
['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']


In [16]:
# 시나리오 문항을 2개씩 묶어서 5개 축으로
# (문항1, 문항2) -> 합산하여 성향 결정
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

# 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름)
AXES = [
    ("S", "F", "parenting_style_SF"),      # 양육 실행 스타일
    ("A", "H", "education_priority_AH"),   # 교육/성장 우선순위
    ("E", "L", "co_parenting_mode_EL"),    # 공동양육 운영 방식
    ("B", "T", "family_boundary_BT"),      # 확장가족/경계
    ("P", "G", "planning_risk_PG"),        # 계획/리스크 대응
]

print(f"축 개수: {len(AXES)}, 페어 개수: {len(pairs)}")
for i, (pair, axis) in enumerate(zip(pairs, AXES)):
    print(f"  축 {i+1} ({axis[2]}): {pair[0]} + {pair[1]}")

축 개수: 5, 페어 개수: 5
  축 1 (parenting_style_SF): sc_toothbrushing + sc_bedtime_story
  축 2 (education_priority_AH): sc_competition_2nd + sc_talent_education
  축 3 (co_parenting_mode_EL): sc_discipline_conflict + sc_play_vs_chores
  축 4 (family_boundary_BT): sc_grandparents_help + sc_inlaws_advice
  축 5 (planning_risk_PG): sc_rainy_zoo + sc_education_fund_risk


In [17]:
def calculate_trait_scores(df, pairs, axes):
    """
    시나리오 응답으로부터 성향 점수 계산
    응답: 1-5점 (1=왼쪽 성향, 5=오른쪽 성향, 3=중립)
    
    Returns:
    --------
    trait_df: 각 축의 연속 점수 (0~1 범위로 정규화)
    mbti_df: 각 축의 문자 표현 (S/F, A/H 등)
    """
    trait_scores = {}
    mbti_letters = {}
    
    for (col1, col2), (left, right, axis_name) in zip(pairs, axes):
        if col1 not in df.columns or col2 not in df.columns:
            continue
        
        # 응답값 합산 (결측값은 3으로 대체)
        v1 = df[col1].fillna(3)
        v2 = df[col2].fillna(3)
        
        # 두 문항 합산 (2~10점 범위)
        total = v1 + v2
        
        # 0~1 범위로 정규화
        trait_scores[axis_name] = (total - 2) / 8.0
        
        # MBTI 스타일 문자 결정
        # 합이 6 미만이면 왼쪽(S, A, E, B, P), 6 초과면 오른쪽(F, H, L, T, G)
        mbti_letters[axis_name + '_letter'] = total.apply(
            lambda x: left if x < 6 else (right if x > 6 else left)  # 중립은 왼쪽으로
        )
    
    return pd.DataFrame(trait_scores), pd.DataFrame(mbti_letters)

df_traits, df_mbti = calculate_trait_scores(df, pairs, AXES)
print(f"성향 점수 shape: {df_traits.shape}")
df_traits.head()

성향 점수 shape: (34, 5)


,parenting_style_SF,education_priority_AH,co_parenting_mode_EL,family_boundary_BT,planning_risk_PG
user_name,,,,,
강현준,0.500,0.875,0.750,0.500,0.875
임경빈,0.625,0.500,0.750,0.750,0.625
이정미,0.875,0.875,0.875,0.500,0.500
쳌,0.625,0.875,0.750,0.500,0.625
김용희,0.375,0.500,0.375,0.625,0.375


In [18]:
# MBTI 유형 조합 생성
def create_mbti_type(row):
    letters = []
    for col in df_mbti.columns:
        if pd.notna(row[col]):
            letters.append(row[col])
    return ''.join(letters)

df_mbti['childcare_mbti'] = df_mbti.apply(create_mbti_type, axis=1)
print("자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts())

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    4
SHLBP    3
FHEBP    3
SHEBG    3
FHLTG    2
SHETG    2
SALBP    2
SHEBP    2
FALTG    1
FHLBP    1
FHLBG    1
SAETP    1
SAEBG    1
SHLTG    1
SALBG    1
SHETP    1
FHEBG    1
FHETG    1
FHLTP    1
SALTP    1
SHLTP    1
Name: count, dtype: int64


## (4) 모든 피처 합치기

In [19]:
# 모든 피처 합치기
df_features = pd.concat([df_onehot, df_traits], axis=1)

# 결측값 처리
df_features = df_features.fillna(0)

print(f"최종 피처 매트릭스 shape: {df_features.shape}")
print(f"\n피처 목록:")
for i, col in enumerate(df_features.columns):
    print(f"  {i+1}. {col}")

최종 피처 매트릭스 shape: (34, 34)

피처 목록:
  1. p_children_count_1명
  2. p_children_count_2명
  3. p_children_count_3명
  4. p_children_count_그 이상
  5. p_children_count_기타
  6. p_children_composition_오직 딸
  7. p_children_composition_오직 아들
  8. p_children_composition_딸 1명, 아들 1명
  9. p_children_composition_기타
  10. p_children_timing_결혼 즉시
  11. p_children_timing_결혼 후 1~2년 이내
  12. p_children_timing_결혼 후 3~5년 이내
  13. p_children_timing_경제적 안정 후
  14. p_children_timing_기타
  15. p_infertility_alternative_의학적 도움 적극 시도
  16. p_infertility_alternative_입양 고려
  17. p_infertility_alternative_무자녀
  18. p_infertility_alternative_기타
  19. e_childcare_cost_share_노후보단 자녀 교육
  20. e_childcare_cost_share_노후 먼저, 남는 예산으로 지원
  21. e_childcare_cost_share_기타
  22. e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아
  23. e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)
  24. e_parental_leave_burden_기타
  25. child_values_open_경제적 성공, 사회적 지위
  26. child_values_open_도덕적, 타인 배려
  27. child_values_open_자신이 좋아하는 일, 행복
  28. chil

In [20]:
# 피처 그룹 정의 (추천 시스템에서 사용)
FEATURE_GROUPS = {
    'value_cols': [c for c in df_features.columns if c.startswith(('p_', 'e_', 'child_'))],
    'trait_cols': [c for c in df_features.columns if c in ['parenting_style_SF', 'education_priority_AH', 
                                                           'co_parenting_mode_EL', 'family_boundary_BT', 
                                                           'planning_risk_PG']]
}

print(f"가치관 피처 수: {len(FEATURE_GROUPS['value_cols'])}")
print(f"성향 피처 수: {len(FEATURE_GROUPS['trait_cols'])}")

가치관 피처 수: 29
성향 피처 수: 5


# 4. 유사도 기반 매칭 추천

코사인 유사도를 사용하여 가치관이 비슷한 사람을 추천합니다.

In [21]:
# 가치관 유사도 매트릭스 계산
value_cols = FEATURE_GROUPS['value_cols']
if len(value_cols) > 0:
    val_sim_matrix = cosine_similarity(df_features[value_cols])
    val_sim_df = pd.DataFrame(val_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"가치관 유사도 매트릭스 shape: {val_sim_df.shape}")
else:
    val_sim_df = pd.DataFrame(index=df_features.index, columns=df_features.index)

val_sim_df.head()

가치관 유사도 매트릭스 shape: (34, 34)


user_name,강현준,임경빈,이정미,쳌,김용희,김진우,김경표,임태나,이나원,김종우,...,ㅁㅎ,강우엉,임세희,하준우,샌새,이지현,최서진,이은수,민채영,유현희
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,1.000000,0.582160,0.356235,0.516388,0.596354,0.152333,0.342748,0.318952,0.328358,0.247540,...,0.548529,0.086119,0.049036,0.672032,0.367767,0.162871,0.403034,0.628372,0.260157,0.071970
임경빈,0.582160,1.000000,0.534258,0.666667,0.696311,0.167984,0.167984,0.281379,0.061633,0.335968,...,0.645214,0.101298,0.108148,0.592865,0.504689,0.498904,0.533333,0.839921,0.229510,0.428571
이정미,0.356235,0.534258,1.000000,0.353553,0.492366,0.160357,0.267261,0.562786,0.313786,0.160357,...,0.527930,0.483494,0.390007,0.411597,0.504715,0.355600,0.282843,0.374166,0.109545,0.646498
쳌,0.516388,0.666667,0.353553,1.000000,0.783349,0.566947,0.283473,0.361773,0.312019,0.377964,...,0.777714,0.227921,0.000000,0.545705,0.405554,0.314309,0.400000,0.944911,0.710047,0.214286
김용희,0.596354,0.696311,0.492366,0.783349,1.000000,0.328976,0.460566,0.503812,0.531085,0.592157,...,0.992806,0.317408,0.169435,0.548860,0.564782,0.187592,0.661495,0.855337,0.584307,0.298419


In [22]:
# 성향 유사도 매트릭스 계산
trait_cols = FEATURE_GROUPS['trait_cols']
if len(trait_cols) > 0:
    trait_sim_matrix = cosine_similarity(df_features[trait_cols])
    trait_sim_df = pd.DataFrame(trait_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"성향 유사도 매트릭스 shape: {trait_sim_df.shape}")
else:
    trait_sim_df = pd.DataFrame(index=df_features.index, columns=df_features.index)

trait_sim_df.head()

성향 유사도 매트릭스 shape: (34, 34)


user_name,강현준,임경빈,이정미,쳌,김용희,김진우,김경표,임태나,이나원,김종우,...,ㅁㅎ,강우엉,임세희,하준우,샌새,이지현,최서진,이은수,민채영,유현희
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,1.000000,0.944806,0.945599,0.985331,0.931809,0.970907,0.947168,0.972698,0.968540,0.967973,...,0.962427,0.925156,0.787570,0.942204,0.883418,0.963003,0.722393,0.960125,0.964929,0.948636
임경빈,0.944806,1.000000,0.948026,0.955985,0.970362,0.917417,0.849946,0.865928,0.968257,0.900222,...,0.953407,0.842701,0.705886,0.931735,0.891673,0.948026,0.851298,0.930746,0.974477,0.963364
이정미,0.945599,0.948026,1.000000,0.985372,0.924526,0.978139,0.829205,0.918464,0.932706,0.855843,...,0.949243,0.853177,0.815519,0.959406,0.838405,0.983240,0.821479,0.930568,0.993631,0.974442
쳌,0.985331,0.955985,0.985372,1.000000,0.947389,0.986666,0.902817,0.953642,0.961587,0.925885,...,0.968408,0.904256,0.834641,0.955505,0.872562,0.991454,0.781598,0.967716,0.991687,0.979903
김용희,0.931809,0.970362,0.924526,0.947389,1.000000,0.886844,0.840841,0.828965,0.964209,0.893281,...,0.958016,0.881993,0.820347,0.856728,0.920184,0.951718,0.792482,0.977501,0.955330,0.966158


In [23]:
def get_similarity_recommendations(user_name, top_n=5, value_weight=0.7, trait_weight=0.3):
    """
    유사도 기반 추천: 가치관과 성향이 비슷한 사람 추천
    
    Parameters:
    -----------
    user_name : str - 추천 대상 사용자
    top_n : int - 추천할 인원 수
    value_weight : float - 가치관 유사도 가중치
    trait_weight : float - 성향 유사도 가중치
    
    Returns:
    --------
    DataFrame with recommendations
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    recommendations = []
    
    for other_user in df_features.index:
        if user_name == other_user:
            continue
        
        # 가치관 유사도
        val_score = val_sim_df.loc[user_name, other_user] if user_name in val_sim_df.index else 0
        
        # 성향 유사도
        trait_score = trait_sim_df.loc[user_name, other_user] if user_name in trait_sim_df.index else 0
        
        # 종합 점수
        total_score = value_weight * val_score + trait_weight * trait_score
        
        # 이상형 매칭 여부 확인
        if 'ideal_type' in df.columns:
            my_ideal = df.loc[user_name, 'ideal_type'] if user_name in df.index else None
            other_ideal = df.loc[other_user, 'ideal_type'] if other_user in df.index else None
            ideal_match = my_ideal == other_ideal if my_ideal and other_ideal else False
        else:
            ideal_match = False
        
        # MBTI 유형 가져오기
        other_mbti = df_mbti.loc[other_user, 'childcare_mbti'] if other_user in df_mbti.index else 'N/A'
        
        recommendations.append({
            'name': other_user,
            'value_similarity': round(val_score, 4),
            'trait_similarity': round(trait_score, 4),
            'total_score': round(total_score, 4),
            'childcare_mbti': other_mbti,
            'ideal_type_match': ideal_match,
            'match_type': '유사형(Similar)'
        })
    
    result_df = pd.DataFrame(recommendations)
    result_df = result_df.sort_values('total_score', ascending=False).head(top_n)
    result_df = result_df.reset_index(drop=True)
    result_df.index = result_df.index + 1  # 1부터 시작
    
    return result_df

# 테스트
test_user = df_features.index[0]
print(f"\n=== {test_user}님의 유사도 기반 추천 ===")
get_similarity_recommendations(test_user, top_n=5)


=== 강현준님의 유사도 기반 추천 ===


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,하준우,0.6720,0.9422,0.7531,SALBP,False,유사형(Similar)
2,언리얼/이동진,0.6389,0.9538,0.7334,SHETG,False,유사형(Similar)
3,이은수,0.6284,0.9601,0.7279,SHETG,False,유사형(Similar)
4,김용희,0.5964,0.9318,0.6970,SAETP,False,유사형(Similar)
5,임경빈,0.5822,0.9448,0.6910,FALTG,False,유사형(Similar)


# 5. 보완도 기반 매칭 추천

**핵심 가치관은 유사하면서, 양육 스타일은 서로 보완하는 사람을 추천합니다.**

- 가치관 (자녀 수, 교육비, 자녀 가치관 등) → **유사할수록 좋음**
- 양육 성향 (SF, AH, EL, BT, PG 축) → **서로 다르면 보완적**

In [24]:
def calculate_complementary_score(user1, user2, df_features, trait_cols):
    """
    보완도 점수 계산
    성향 점수의 차이가 클수록 보완도가 높음
    단, 너무 극단적인 차이(반대)는 갈등 가능성이 있으므로 적절한 차이가 최적
    
    Returns:
    --------
    complementary_score : float (0~1, 높을수록 보완적)
    trait_details : dict (각 축별 차이 상세)
    """
    if len(trait_cols) == 0:
        return 0, {}
    
    user1_traits = df_features.loc[user1, trait_cols]
    user2_traits = df_features.loc[user2, trait_cols]
    
    # 각 축별 차이 계산
    trait_details = {}
    complementary_scores = []
    
    for col in trait_cols:
        diff = abs(user1_traits[col] - user2_traits[col])
        
        # 보완도 점수: 차이가 0.3~0.5일 때 가장 높음 (적절한 보완)
        # 차이가 0이면 동질, 차이가 1이면 극단적 반대
        # 가우시안 형태로 0.4에서 피크
        optimal_diff = 0.4
        comp_score = np.exp(-((diff - optimal_diff) ** 2) / 0.1)
        
        complementary_scores.append(comp_score)
        trait_details[col] = {
            'user1': round(user1_traits[col], 3),
            'user2': round(user2_traits[col], 3),
            'diff': round(diff, 3),
            'comp_score': round(comp_score, 3)
        }
    
    avg_complementary = np.mean(complementary_scores) if complementary_scores else 0
    
    return avg_complementary, trait_details

In [25]:
def get_complementary_recommendations(user_name, top_n=5, value_threshold=0.5):
    """
    보완도 기반 추천: 가치관은 유사하면서 성향은 보완적인 사람 추천
    
    Parameters:
    -----------
    user_name : str - 추천 대상 사용자
    top_n : int - 추천할 인원 수
    value_threshold : float - 최소 가치관 유사도 (이 이상이어야 추천 대상)
    
    Returns:
    --------
    DataFrame with recommendations
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    trait_cols = FEATURE_GROUPS['trait_cols']
    recommendations = []
    
    # 내 MBTI 가져오기
    my_mbti = df_mbti.loc[user_name, 'childcare_mbti'] if user_name in df_mbti.index else 'N/A'
    
    for other_user in df_features.index:
        if user_name == other_user:
            continue
        
        # 가치관 유사도 (기본 필터)
        val_score = val_sim_df.loc[user_name, other_user] if user_name in val_sim_df.index else 0
        
        # 가치관 유사도가 기준 이하면 스킵
        if val_score < value_threshold:
            continue
        
        # 보완도 계산
        comp_score, trait_details = calculate_complementary_score(
            user_name, other_user, df_features, trait_cols
        )
        
        # 종합 점수: 가치관 유사도 * 보완도
        total_score = val_score * comp_score
        
        # MBTI 유형 가져오기
        other_mbti = df_mbti.loc[other_user, 'childcare_mbti'] if other_user in df_mbti.index else 'N/A'
        
        # 이상형 매칭 여부
        if 'ideal_type' in df.columns:
            my_ideal = df.loc[user_name, 'ideal_type'] if user_name in df.index else None
            other_ideal = df.loc[other_user, 'ideal_type'] if other_user in df.index else None
            ideal_match = my_ideal == other_ideal if my_ideal and other_ideal else False
        else:
            ideal_match = False
        
        recommendations.append({
            'name': other_user,
            'value_similarity': round(val_score, 4),
            'complementary_score': round(comp_score, 4),
            'total_score': round(total_score, 4),
            'my_mbti': my_mbti,
            'their_mbti': other_mbti,
            'ideal_type_match': ideal_match,
            'match_type': '보완형(Complementary)'
        })
    
    if not recommendations:
        return f"가치관 유사도 {value_threshold} 이상인 매칭 대상이 없습니다."
    
    result_df = pd.DataFrame(recommendations)
    result_df = result_df.sort_values('total_score', ascending=False).head(top_n)
    result_df = result_df.reset_index(drop=True)
    result_df.index = result_df.index + 1
    
    return result_df

# 테스트
print(f"\n=== {test_user}님의 보완도 기반 추천 ===")
get_complementary_recommendations(test_user, top_n=5, value_threshold=0.3)


=== 강현준님의 보완도 기반 추천 ===


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,하준우,0.6720,0.6915,0.4647,SHLBG,SALBP,False,보완형(Complementary)
2,김용희,0.5964,0.7662,0.4570,SHLBG,SAETP,False,보완형(Complementary)
3,이주헌,0.4446,0.8801,0.3913,SHLBG,FHEBP,False,보완형(Complementary)
4,이은수,0.6284,0.6134,0.3854,SHLBG,SHETG,False,보완형(Complementary)
5,임경빈,0.5822,0.6524,0.3798,SHLBG,FALTG,False,보완형(Complementary)


# 6. 통합 추천 시스템

유사도 기반과 보완도 기반 추천을 모두 제공하는 통합 함수

In [26]:
def get_smart_recommendations(user_name, top_n=5, mode='both'):
    """
    스마트 추천 시스템: 유사도 + 보완도 기반 통합 추천
    
    Parameters:
    -----------
    user_name : str - 추천 대상 사용자
    top_n : int - 각 유형별 추천할 인원 수
    mode : str - 'similar', 'complementary', 'both'
    
    Returns:
    --------
    dict with recommendation DataFrames
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    results = {}
    
    # 사용자 정보 출력
    print("="*60)
    print(f"🎯 {user_name}님을 위한 맞춤 추천")
    print("="*60)
    
    # 사용자 프로필
    if user_name in df_mbti.index:
        user_mbti = df_mbti.loc[user_name, 'childcare_mbti']
        print(f"\n📋 당신의 자녀양육 MBTI: {user_mbti}")
    
    if 'ideal_type' in df.columns and user_name in df.index:
        user_ideal = df.loc[user_name, 'ideal_type']
        print(f"💕 당신의 이상형: {user_ideal}")
    
    # 유사도 기반 추천
    if mode in ['similar', 'both']:
        print(f"\n\n🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람")
        print("-"*50)
        similar_recs = get_similarity_recommendations(user_name, top_n)
        results['similar'] = similar_recs
        if isinstance(similar_recs, pd.DataFrame):
            display(similar_recs)
        else:
            print(similar_recs)
    
    # 보완도 기반 추천
    if mode in ['complementary', 'both']:
        print(f"\n\n🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람")
        print("-"*50)
        comp_recs = get_complementary_recommendations(user_name, top_n, value_threshold=0.3)
        results['complementary'] = comp_recs
        if isinstance(comp_recs, pd.DataFrame):
            display(comp_recs)
        else:
            print(comp_recs)
    
    print("\n" + "="*60)
    
    return results

In [27]:
# 통합 추천 테스트
test_user = df_features.index[0]
results = get_smart_recommendations(test_user, top_n=5, mode='both')

🎯 강현준님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: SHLBG
💕 당신의 이상형: 꽃돼지상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,하준우,0.6720,0.9422,0.7531,SALBP,False,유사형(Similar)
2,언리얼/이동진,0.6389,0.9538,0.7334,SHETG,False,유사형(Similar)
3,이은수,0.6284,0.9601,0.7279,SHETG,False,유사형(Similar)
4,김용희,0.5964,0.9318,0.6970,SAETP,False,유사형(Similar)
5,임경빈,0.5822,0.9448,0.6910,FALTG,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,하준우,0.6720,0.6915,0.4647,SHLBG,SALBP,False,보완형(Complementary)
2,김용희,0.5964,0.7662,0.4570,SHLBG,SAETP,False,보완형(Complementary)
3,이주헌,0.4446,0.8801,0.3913,SHLBG,FHEBP,False,보완형(Complementary)
4,이은수,0.6284,0.6134,0.3854,SHLBG,SHETG,False,보완형(Complementary)
5,임경빈,0.5822,0.6524,0.3798,SHLBG,FALTG,False,보완형(Complementary)


In [33]:
# 여러 사용자 테스트
print("\n" + "#"*70)
print("# 다른 사용자 추천 결과")
print("#"*70)

for user in df_features.index[1:4]:  # 2~4번째 사용자
    print("\n")
    get_smart_recommendations(user, top_n=3, mode='both')


######################################################################
# 다른 사용자 추천 결과
######################################################################


🎯 임경빈님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FALTG
💕 당신의 이상형: 꼬부기상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,이은수,0.8399,0.9307,0.8672,SHETG,False,유사형(Similar)
2,언리얼/이동진,0.7800,0.9635,0.8351,SHETG,False,유사형(Similar)
3,이주헌,0.8280,0.8048,0.8211,FHEBP,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,이주헌,0.8280,0.6050,0.5009,FALTG,FHEBP,False,보완형(Complementary)
2,언리얼/이동진,0.7800,0.6011,0.4688,FALTG,SHETG,False,보완형(Complementary)
3,김용희,0.6963,0.6524,0.4543,FALTG,SAETP,False,보완형(Complementary)





🎯 이정미님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FHLBP
💕 당신의 이상형: 곰상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


...

,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,이주헌,0.7766,0.6737,0.5232,FHLBP,FHEBP,False,보완형(Complementary)
2,임태나,0.5628,0.7305,0.4111,FHLBP,SHLBG,False,보완형(Complementary)
3,임경빈,0.5343,0.7059,0.3771,FHLBP,FALTG,False,보완형(Complementary)





🎯 쳌님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FHLBG
💕 당신의 이상형: 고양이상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,이은수,0.9449,0.9677,0.9518,SHETG,False,유사형(Similar)
2,박재혁,0.8504,0.9782,0.8888,SALBG,True,유사형(Similar)
3,ㅁㅎ,0.7777,0.9684,0.8349,FHEBG,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,김용희,0.7833,0.8108,0.6351,FHLBG,SAETP,False,보완형(Complementary)
2,이은수,0.9449,0.5476,0.5174,FHLBG,SHETG,False,보완형(Complementary)
3,민채영,0.7100,0.7059,0.5012,FHLBG,FHLTG,False,보완형(Complementary)


## 상세 분석: 두 사용자 간 매칭 상세 정보

In [29]:
def analyze_match_detail(user1, user2):
    """
    두 사용자 간의 매칭 상세 분석
    """
    if user1 not in df_features.index or user2 not in df_features.index:
        return "사용자를 찾을 수 없습니다."
    
    print("="*60)
    print(f"📊 {user1} ↔ {user2} 매칭 상세 분석")
    print("="*60)
    
    # 1. 가치관 유사도
    val_score = val_sim_df.loc[user1, user2]
    print(f"\n🎯 가치관 유사도: {val_score:.2%}")
    
    # 2. 성향 비교
    trait_cols = FEATURE_GROUPS['trait_cols']
    if trait_cols:
        print(f"\n📈 양육 성향 비교:")
        print("-"*50)
        print(f"{'축':<25} {user1:<10} {user2:<10} {'차이':>10}")
        print("-"*50)
        
        for col in trait_cols:
            v1 = df_features.loc[user1, col]
            v2 = df_features.loc[user2, col]
            diff = abs(v1 - v2)
            print(f"{col:<25} {v1:.3f}      {v2:.3f}      {diff:.3f}")
    
    # 3. MBTI 비교
    if user1 in df_mbti.index and user2 in df_mbti.index:
        mbti1 = df_mbti.loc[user1, 'childcare_mbti']
        mbti2 = df_mbti.loc[user2, 'childcare_mbti']
        print(f"\n🧬 자녀양육 MBTI:")
        print(f"   {user1}: {mbti1}")
        print(f"   {user2}: {mbti2}")
        
        # 같은 문자 개수
        same_count = sum(1 for a, b in zip(mbti1, mbti2) if a == b)
        print(f"   일치하는 축: {same_count}/5")
    
    # 4. 이상형 비교
    if 'ideal_type' in df.columns:
        ideal1 = df.loc[user1, 'ideal_type'] if user1 in df.index else 'N/A'
        ideal2 = df.loc[user2, 'ideal_type'] if user2 in df.index else 'N/A'
        print(f"\n💕 이상형:")
        print(f"   {user1}: {ideal1}")
        print(f"   {user2}: {ideal2}")
    
    # 5. 종합 평가
    comp_score, _ = calculate_complementary_score(user1, user2, df_features, trait_cols)
    print(f"\n🏆 종합 평가:")
    print(f"   유사도 점수: {val_score:.2%}")
    print(f"   보완도 점수: {comp_score:.2%}")
    
    if val_score >= 0.7 and comp_score >= 0.5:
        print(f"   → ⭐⭐⭐ 최고의 매칭! (가치관 일치 + 적절한 보완)")
    elif val_score >= 0.5:
        print(f"   → ⭐⭐ 좋은 매칭 (가치관 유사)")
    elif comp_score >= 0.5:
        print(f"   → ⭐⭐ 보완적 매칭 (성향 보완)")
    else:
        print(f"   → ⭐ 보통 매칭")
    
    print("="*60)

# 테스트
users = list(df_features.index[:2])
if len(users) >= 2:
    analyze_match_detail(users[0], users[1])

📊 강현준 ↔ 임경빈 매칭 상세 분석

🎯 가치관 유사도: 58.22%

📈 양육 성향 비교:
--------------------------------------------------
축                         강현준        임경빈                차이
--------------------------------------------------
parenting_style_SF        0.500      0.625      0.125
education_priority_AH     0.875      0.500      0.375
co_parenting_mode_EL      0.750      0.750      0.000
family_boundary_BT        0.500      0.750      0.250
planning_risk_PG          0.875      0.625      0.250

🧬 자녀양육 MBTI:
   강현준: SHLBG
   임경빈: FALTG
   일치하는 축: 2/5

💕 이상형:
   강현준: 꽃돼지상
   임경빈: 꼬부기상

🏆 종합 평가:
   유사도 점수: 58.22%
   보완도 점수: 65.24%
   → ⭐⭐ 좋은 매칭 (가치관 유사)


# 7. 데이터 저장 및 요약

In [30]:
# 처리된 피처 데이터 저장
output_features_path = "processed_features.csv"
df_features.to_csv(output_features_path)
print(f"피처 데이터 저장: {output_features_path}")

# MBTI 데이터 저장
output_mbti_path = "childcare_mbti.csv"
df_mbti.to_csv(output_mbti_path)
print(f"MBTI 데이터 저장: {output_mbti_path}")

# 유사도 매트릭스 저장
output_sim_path = "similarity_matrix.csv"
val_sim_df.to_csv(output_sim_path)
print(f"유사도 매트릭스 저장: {output_sim_path}")

피처 데이터 저장: processed_features.csv
MBTI 데이터 저장: childcare_mbti.csv
유사도 매트릭스 저장: similarity_matrix.csv


In [31]:
print("\n" + "="*70)
print("📊 최종 요약")
print("="*70)
print(f"\n총 사용자 수: {len(df_features)}명")
print(f"피처 수: {df_features.shape[1]}개")
print(f"  - 가치관 피처: {len(FEATURE_GROUPS['value_cols'])}개")
print(f"  - 성향 피처: {len(FEATURE_GROUPS['trait_cols'])}개")

print(f"\n자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts().head(10))

print(f"\n이상형 분포:")
if 'ideal_type' in df.columns:
    print(df['ideal_type'].value_counts())

print("\n" + "="*70)
print("✅ 추천 시스템 구축 완료!")
print("="*70)


📊 최종 요약

총 사용자 수: 34명
피처 수: 34개
  - 가치관 피처: 29개
  - 성향 피처: 5개

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    4
SHLBP    3
FHEBP    3
SHEBG    3
FHLTG    2
SHETG    2
SALBP    2
SHEBP    2
FALTG    1
FHLBP    1
Name: count, dtype: int64

이상형 분포:
ideal_type
여우상     9
곰상      6
고양이상    6
강아지상    5
오리상     2
토끼상     2
꽃돼지상    1
꼬부기상    1
너구리상    1
쿼카상     1
Name: count, dtype: int64

✅ 추천 시스템 구축 완료!
